# Gurobi aviation challenge

In this challenge we look at _tail assignment_, which is the assignment of flights to airplanes, usually in a way to minimize operational costs. This notebook is designed to make it easy to understand the puzzle and provides a single function signature that needs to be implemented by you. We first present data for *flights* and *airplanes*. We then describe what a *solution* looks like - essentially one *schedule* per airplane. Finally the challenge for you is to create a function that takes a collection of flights and airplanes and creates a solution that is both valid and minimizes cost. Hint - using the *gurobipy* package is your best bet!

While we use *dataclasses* in this challenge, this is not required at all for using Gurobi. You could equally implement the same logic using Pandas dataframes or plain dictionaries. Using an object-oriented approach here makes it more clear what the individual pieces of data mean.

In [1]:
from datetime import datetime, time
from dataclasses import dataclass

## Flights

For this challenge, we start with a collection of flights. For each flight, we have a unique code, its origin and destination airport (single-letter code A through D), departure and arrival time.

In [2]:
@dataclass(frozen=True)
class Flight:
    code: str
    origin: str
    destination: str
    departure: time
    arrival: time

    def duration_hours(self) -> float:
        def to_hours(t: time) -> float:
            return t.hour + t.minute / 60 + t.second / 3600
        return to_hours(self.arrival) - to_hours(self.departure)

    def __lt__(self, other: 'Flight') -> bool:
        return self.departure < other.departure

The puzzle consists of sixteen flights:

In [3]:
def _t(time_str: str) -> time:
    return datetime.strptime(time_str, '%H:%M:%S').time()

flights = [
    Flight('GRB01', 'A', 'B', _t('09:00:00'), _t('10:30:00')),
    Flight('GRB02', 'A', 'D', _t('13:30:00'), _t('14:30:00')),
    Flight('GRB03', 'A', 'C', _t('14:00:00'), _t('17:00:00')),
    Flight('GRB04', 'A', 'B', _t('14:30:00'), _t('16:00:00')),
    Flight('GRB05', 'B', 'C', _t('07:30:00'), _t('10:00:00')),
    Flight('GRB06', 'B', 'D', _t('10:30:00'), _t('13:30:00')),
    Flight('GRB07', 'B', 'A', _t('11:00:00'), _t('12:30:00')),
    Flight('GRB08', 'B', 'D', _t('16:30:00'), _t('19:30:00')),
    Flight('GRB09', 'B', 'C', _t('17:30:00'), _t('20:00:00')),
    Flight('GRB10', 'C', 'B', _t('07:30:00'), _t('10:00:00')),
    Flight('GRB11', 'C', 'A', _t('10:00:00'), _t('13:00:00')),
    Flight('GRB12', 'C', 'A', _t('10:30:00'), _t('13:30:00')),
    Flight('GRB13', 'C', 'A', _t('17:00:00'), _t('20:00:00')),
    Flight('GRB14', 'D', 'C', _t('08:00:00'), _t('10:00:00')),
    Flight('GRB15', 'D', 'B', _t('14:00:00'), _t('17:00:00')),
    Flight('GRB16', 'D', 'C', _t('15:00:00'), _t('17:00:00')),
]

## Airplanes

We also have airplanes, each located at an initial airport. For each airplane we have the cost per hour spent flying, as well as the initial airport where the airplane is located at the start of our day.

In [4]:
@dataclass(frozen=True)
class Airplane:
    code: str
    cost_per_hour: int
    start_airport: str

In the challenge we have four airplanes:

In [5]:
airplanes = [
    Airplane('A1', 4000, 'A'),
    Airplane('A3', 8000, 'C'),
    Airplane('A2', 6000, 'B'),
    Airplane('A4', 10000, 'D'),
]

## Solutions

The challenge is to assign all flights to airplanes, minimizing total cost. To represent such an assignment, we first define a `Schedule` class that represents the consecutive flights assigned to a single airplane. In this class, you can see most of the rules of the challenge implemented in a `is_valid` function.

In [6]:

@dataclass
class Schedule:
    airplane: Airplane
    flights: list[Flight]

    def __post_init__(self):
        self.flights.sort()

    def is_valid(self) -> bool:
        if not self.flights:
            return True

        # First flight must start from airplane's start airport
        if self.flights[0].origin != self.airplane.start_airport:
            return False

        # Check consecutive flight connections
        for i in range(len(self.flights) - 1):
            current = self.flights[i]
            next_flight = self.flights[i + 1]

            # Flights must connect: arrival airport = next departure airport
            if current.destination != next_flight.origin:
                return False

            # Previous arrival can't be later than next departure
            if current.arrival > next_flight.departure:
                return False

        return True

    def total_cost(self) -> int:
        return int(sum(
            self.airplane.cost_per_hour * flight.duration_hours()
            for flight in self.flights
        ))

Now we can represent a full assignment as a `Solution` containing a `Schedule` for each `Airplane`.

In [7]:
from collections import defaultdict

@dataclass
class Solution:
    schedules: dict[Airplane, Schedule]

    @classmethod
    def from_assignments(cls, assignments: dict[Flight, Airplane]) -> 'Solution':
        # Group flights by airplane
        airplane_to_flights = defaultdict(list)
        for flight, airplane in assignments.items():
            airplane_to_flights[airplane].append(flight)

        # Create schedules for each airplane
        schedules = {
            airplane: Schedule(airplane, flight_list)
            for airplane, flight_list in airplane_to_flights.items()
        }

        return cls(schedules)

    def is_complete(self, all_flights: list[Flight]) -> bool:
        assigned_flights = {
            flight for schedule in self.schedules.values() for flight in schedule.flights
        }
        return assigned_flights == set(all_flights)

    def validate(self, all_flights: list[Flight]) -> tuple[bool, int]:
        # Check that all flights are assigned
        if not self.is_complete(all_flights):
            return (False, 0)

        # Validate each airplane's schedule and calculate total cost
        total_cost = 0
        for schedule in self.schedules.values():
            if not schedule.is_valid():
                return (False, 0)
            total_cost += schedule.total_cost()

        return (True, total_cost)

## The challenge: Creating the optimal solution
Below is the one missing function, which forms the core of this challenge: creating a valid `Solution` for a given collection of flights and airplanes, which minimizes the total cost. You can call `Solution.validate()` on the result to see if your function generates valid solutions. The bigger challenge is to find the _optimal_ solution!

In [8]:
import gurobipy as gp
from gurobipy import GRB


def solve(flights: list[Flight], airplanes: list[Airplane]) -> Solution:
    model = gp.Model("tail_assignment")
    model.setParam("OutputFlag", 0)

    # Index lookups
    fi = {f: i for i, f in enumerate(flights)}
    ai = {a: i for i, a in enumerate(airplanes)}

    # Precompute compatible consecutive flight pairs:
    # f1 can be followed by f2 if destination matches origin and timing works
    compatible = [
        (f1, f2)
        for f1 in flights
        for f2 in flights
        if f1 is not f2
        and f1.destination == f2.origin
        and f1.arrival <= f2.departure
    ]

    # Decision variables using integer index keys for short names
    # x[i, j] = 1 if flight i is assigned to airplane j
    x = model.addVars(
        [(fi[f], ai[a]) for f in flights for a in airplanes],
        vtype=GRB.BINARY,
        name="x",
    )

    # s[i, j] = 1 if flight i is the FIRST flight on airplane j
    s = model.addVars(
        [(fi[f], ai[a]) for f in flights for a in airplanes],
        vtype=GRB.BINARY,
        name="s",
    )

    # y[i, k, j] = 1 if flight k immediately follows flight i on airplane j
    y = model.addVars(
        [(fi[f1], fi[f2], ai[a]) for (f1, f2) in compatible for a in airplanes],
        vtype=GRB.BINARY,
        name="y",
    )

    # Objective: minimize total cost
    model.setObjective(
        gp.quicksum(
            a.cost_per_hour * f.duration_hours() * x[fi[f], ai[a]]
            for f in flights
            for a in airplanes
        ),
        GRB.MINIMIZE,
    )

    # Constraint 1: Each flight is assigned to exactly one airplane
    for f in flights:
        model.addConstr(
            gp.quicksum(x[fi[f], ai[a]] for a in airplanes) == 1,
        )

    # Constraint 2: Each airplane has at most one first flight
    for a in airplanes:
        model.addConstr(
            gp.quicksum(s[fi[f], ai[a]] for f in flights) <= 1,
        )

    # Constraint 3: First flight must depart from airplane's start airport
    for f in flights:
        for a in airplanes:
            if f.origin != a.start_airport:
                model.addConstr(s[fi[f], ai[a]] == 0)

    # Constraint 4: s[f,a] can only be 1 if x[f,a] = 1
    for f in flights:
        for a in airplanes:
            model.addConstr(s[fi[f], ai[a]] <= x[fi[f], ai[a]])

    # Constraint 5: Flow conservation - for each flight f assigned to airplane a,
    # it must either be the start flight OR preceded by exactly one compatible flight
    for f in flights:
        for a in airplanes:
            predecessors = gp.quicksum(
                y[fi[f1], fi[f], ai[a]] for (f1, f2) in compatible if f2 is f
            )
            model.addConstr(
                s[fi[f], ai[a]] + predecessors == x[fi[f], ai[a]],
            )

    # Constraint 6: Each flight has at most one successor on its airplane
    for f in flights:
        for a in airplanes:
            successors = gp.quicksum(
                y[fi[f], fi[f2], ai[a]] for (f1, f2) in compatible if f1 is f
            )
            model.addConstr(successors <= x[fi[f], ai[a]])

    # Constraint 7: Sequencing only if both flights assigned to same airplane
    for (f1, f2) in compatible:
        for a in airplanes:
            model.addConstr(y[fi[f1], fi[f2], ai[a]] <= x[fi[f1], ai[a]])
            model.addConstr(y[fi[f1], fi[f2], ai[a]] <= x[fi[f2], ai[a]])

    model.optimize()

    # Extract solution
    assignments = {}
    for f in flights:
        for a in airplanes:
            if x[fi[f], ai[a]].X > 0.5:
                assignments[f] = a
                break

    return Solution.from_assignments(assignments)

In [9]:
solution = solve(flights, airplanes)

valid, total_cost = solution.validate(flights)
print(f"Valid: {valid}")
print(f"Total cost: {total_cost}")
print()

for airplane, schedule in sorted(solution.schedules.items(), key=lambda x: x[0].code):
    print(f"Airplane {airplane.code} (${airplane.cost_per_hour}/hr, starts at {airplane.start_airport}):")
    for flight in schedule.flights:
        print(f"  {flight.code}: {flight.origin}->{flight.destination} "
              f"{flight.departure.strftime('%H:%M')}-{flight.arrival.strftime('%H:%M')} "
              f"(${int(airplane.cost_per_hour * flight.duration_hours())})")
    print(f"  Schedule cost: ${schedule.total_cost()}")
    print()

Restricted license - for non-production use only - expires 2027-11-29


Valid: True
Total cost: 257000

Airplane A1 ($4000/hr, starts at A):
  GRB01: A->B 09:00-10:30 ($6000)
  GRB06: B->D 10:30-13:30 ($12000)
  GRB15: D->B 14:00-17:00 ($12000)
  GRB09: B->C 17:30-20:00 ($10000)
  Schedule cost: $40000

Airplane A2 ($6000/hr, starts at B):
  GRB05: B->C 07:30-10:00 ($15000)
  GRB12: C->A 10:30-13:30 ($18000)
  GRB03: A->C 14:00-17:00 ($18000)
  GRB13: C->A 17:00-20:00 ($18000)
  Schedule cost: $69000

Airplane A3 ($8000/hr, starts at C):
  GRB10: C->B 07:30-10:00 ($20000)
  GRB07: B->A 11:00-12:30 ($12000)
  GRB04: A->B 14:30-16:00 ($12000)
  GRB08: B->D 16:30-19:30 ($24000)
  Schedule cost: $68000

Airplane A4 ($10000/hr, starts at D):
  GRB14: D->C 08:00-10:00 ($20000)
  GRB11: C->A 10:00-13:00 ($30000)
  GRB02: A->D 13:30-14:30 ($10000)
  GRB16: D->C 15:00-17:00 ($20000)
  Schedule cost: $80000



## Good luck!